# Error Case 20 — 401: Credential Rotation Required

**Error:** `401 Unauthorized`  
**Root cause:** Principal created with `credentialRotationRequired=true` — must rotate before use  
**Fix:** Rotate credentials immediately after principal creation

## Flow
```
Admin creates principal with credentialRotationRequired=true
  → ETL engineer gets credentials
  → ETL calls POST /oauth/tokens → 200 OK (token issued)
  → ETL calls GET /catalogs with token → 401
  → Message: credential rotation required
  → Must rotate credentials first, then retry
```

## Why this matters in production
```
Security policy: force credential change on first use
Common pattern: admin creates principal, hands to team
Team uses credentials without rotating → all requests fail
Confusing: token issued successfully but API calls fail
```

In [ ]:
# ── src path bootstrap (shared modules live in /src) ────────────────────────────
import sys, pathlib
_ROOT = pathlib.Path.cwd()
while not (_ROOT / "src").is_dir() and _ROOT != _ROOT.parent:
    _ROOT = _ROOT.parent
_SRC = _ROOT / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

import sys
import importlib, nb_support
importlib.reload(nb_support)
from nb_support import *
require_not_prod("error-case test")  # PROD-safe: refuses to run against company PROD

cleanup()
print('✅ Setup complete')

In [ ]:
# Cell 2: Create principal WITH credentialRotationRequired=true
r = requests.post(f'{BASE_MGMT}/principals', headers=h(), json={
    'principal': {'name': TEST_PRINCIPAL, 'type': 'SERVICE'},
    'credentialRotationRequired': True,   # ← force rotation
})
print_response(r, 'Create principal with rotation required')
assert r.status_code == 201

cred = r.json()['credentials']
print(f'\nCredentials issued:')
print(f'  clientId:     {cred["clientId"]}')
print(f'  clientSecret: {cred["clientSecret"]}')

In [ ]:
# Cell 3: Get token — succeeds (token issued)
r_tok = get_token(cred['clientId'], cred['clientSecret'])
print_response(r_tok, 'Token request (should succeed)')
assert r_tok.status_code == 200
principal_token = r_tok.json()['access_token']
print('\n✅ Token obtained — but rotation still required!')

In [ ]:
# Cell 4: Use token — 401 because rotation required
r = requests.get(
    f'{BASE_MGMT}/catalogs',
    headers={'Authorization': f'Bearer {principal_token}', 'Polaris-Realm': REALM}
)
print_response(r, '401 Credential rotation required')
assert r.status_code == 401
print('\n✅ 401 confirmed — must rotate before use')
print('\nKey insight:')
print('  Token issued ✅ but API calls rejected ❌')
print('  Confusing pattern — token looks valid')

In [ ]:
# Cell 5: Find logs in OpenSearch
wait_for_logs(10)
print('Searching for credential rotation 401 errors...')
hits = search_401_errors(minutes_ago=1)
print_logs(hits)

In [ ]:
# Cell 6: Fix — rotate credentials before use
print('Fix: Rotate credentials using INITIAL credentials...')

# Rotation must use principal's OWN token
r_rotate = requests.post(
    f'{BASE_MGMT}/principals/{TEST_PRINCIPAL}/credentials:rotate',
    headers={
        'Authorization': f'Bearer {principal_token}',
        'Polaris-Realm': REALM,
        'Content-Type': 'application/json',
    }
)
print_response(r_rotate, 'Credential rotation')

if r_rotate.status_code == 200:
    new_cred = r_rotate.json()['credentials']
    print(f'\nNew credentials:')
    print(f'  clientId:     {new_cred["clientId"]}')
    print(f'  clientSecret: {new_cred["clientSecret"]}')
    
    # Get new token with rotated credentials
    r_new_tok = get_token(new_cred['clientId'], new_cred['clientSecret'])
    new_token = r_new_tok.json()['access_token']
    
    # Verify API works now
    r_verify = requests.get(
        f'{BASE_MGMT}/catalogs',
        headers={'Authorization': f'Bearer {new_token}', 'Polaris-Realm': REALM}
    )
    print(f'\nAfter rotation: {r_verify.status_code}')
    assert r_verify.status_code == 200
    print('\n✅ Fixed — rotated credentials work')
else:
    print(f'Rotation failed: {r_rotate.status_code}')

cleanup()
print('\n=== Summary ===')
print('Error:    401 even with valid token')
print('Cause:    credentialRotationRequired=true on principal')
print('Detect:   401 after successful token — check rotation flag')
print('Fix:      POST /principals/{name}/credentials:rotate')
print('         Use principal OWN token to rotate (not admin token)')
print('Process:  Create → get token → rotate → get new token → use')